# CS701: team SC2 pipeline screens (workers T1 / T2)
Junkai's SC2 code (DINOv3 ViT-B + LoRA r80 + Cascade, `team/`, copied unchanged from the 51.86 backup) on the team's
local 80:10:10 split, 36 epochs. This copy is lane **T2**. Colab gives each notebook file one runtime, so lane T2 needs its own file. Run all.
Queue: `colab/team-queue.json` (separate from A/B/S). T1 runs the smoke first; T2 waits for it.
Runs go to `MyDrive/cs701-runs/team/`, logs and metrics are pushed to `results/team-*/`.

In [ ]:
REPO = 'luat2003/CS701-SAR'
WORKER = 'T2'  # this file is lane T2; worker-team.ipynb is lane T1 (two files = two separate runtimes)
assert WORKER in ('T1', 'T2')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, sys
os.environ['CS701_GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['GIT_TERMINAL_PROMPT'] = '0'
askpass = pathlib.Path('/content/.cs701-team-askpass.py')
askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["CS701_GH_TOKEN"])\n')
askpass.chmod(0o700)
os.environ['GIT_ASKPASS'] = str(askpass)
repo = pathlib.Path('/content/work-team')
url = f'https://github.com/{REPO}.git'
if not repo.exists():
    subprocess.run(['git', 'clone', '-q', url, str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'config', 'credential.helper', ''], check=True)
subprocess.run(['git', 'config', 'user.name', f'colab-team-{WORKER}'], check=True)
subprocess.run(['git', 'config', 'user.email', 'colab-worker@users.noreply.github.com'], check=True)
subprocess.run(['git', 'pull', '--rebase', '-q'], check=True)
subprocess.run(['git', 'log', '-1', '--format=%h %s'], check=True)

In [ ]:
# The team's stack: torch 2.9.0 (cu128), torchvision 0.24.0, timm 1.0.24, pycocotools 2.0.11.
# terratorch (only used for its TF32 switch, which sarbench/backbones.py sets itself without it) is optional on Python 3.13.
def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    print('pip', ' '.join(args), '->', 'OK' if r.returncode == 0 else 'FAILED')
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-3000:])
    return r.returncode == 0
pip('terratorch==1.2.7', 'torchgeo==0.9.0')  # optional; installed first so the pins below win
assert pip('torch==2.9.0', 'torchvision==0.24.0', '--index-url', 'https://download.pytorch.org/whl/cu128')
assert pip('timm==1.0.24', 'pycocotools==2.0.11')
check = subprocess.run([sys.executable, '-c', """import torch, timm, torchvision
print('python', __import__('sys').version.split()[0], 'torch', torch.__version__, 'torchvision', torchvision.__version__, 'timm', timm.__version__)
try:
    import terratorch; print('terratorch', terratorch.__version__)
except Exception as e:
    print('terratorch not available (TF32 set directly):', type(e).__name__, e)
assert torch.__version__.startswith('2.9.0'), torch.__version__
assert torchvision.__version__.startswith('0.24.0'), torchvision.__version__
assert timm.__version__ == '1.0.24', timm.__version__
assert torch.cuda.is_available(), 'no GPU'
print(torch.cuda.get_device_name())
import sys; sys.path.insert(0, 'team'); import sarbench.backbones, sarbench.model; print('team code imports OK')"""], capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0, 'environment check failed: see the message above'

In [ ]:
# Course data (cached on Drive), then the team's split: 7512 train / 940 val / 940 test (seed 0, as SC1)
import hashlib, json, tarfile
cache = pathlib.Path('/content/drive/MyDrive/cs701-data')
pathlib.Path('data').mkdir(exist_ok=True)
for split_name in ('train', 'val'):
    if not pathlib.Path('data', split_name).exists():
        with tarfile.open(cache / f'{split_name}.tar.gz') as tar:
            tar.extractall('data', filter='data')
split = json.loads(pathlib.Path('artifacts/split.json').read_text())
for name, key in (('instances.json', 'annotation_sha256'), ('labels.csv', 'labels_sha256')):
    assert hashlib.sha256(pathlib.Path('data/train', name).read_bytes()).hexdigest() == split[key], name
out = subprocess.run([sys.executable, 'team/prepare_course_data.py', '--src', 'data', '--dst', '/content/team-data/SARFact-Course-20K'],
                     check=True, capture_output=True, text=True).stdout
print(out)
assert 'train: 7512 images' in out and 'val: 940 images' in out and 'test: 940 images' in out, 'split differs from SC1'

In [ ]:
# Checkpoints and raw dumps live on Drive; runs/ in the repo is a symlink to it
target = pathlib.Path('/content/drive/MyDrive/cs701-runs')
target.mkdir(parents=True, exist_ok=True)
if not pathlib.Path('runs').is_symlink():
    assert not pathlib.Path('runs').exists(), 'runs/ is a real folder'
    pathlib.Path('runs').symlink_to(target, target_is_directory=True)
for job in json.loads(pathlib.Path('colab/team-queue.json').read_text()):
    if job['worker'] == WORKER:
        print(job['name'], ' '.join(job['cmd'][2:]))

The worker runs the jobs of this lane one by one and pushes logs every 10 min. A dropped runtime: rerun the notebook;
an interrupted 36-epoch screen restarts from scratch (the team code has no resume). It waits 60 min for new jobs, then releases the GPU.

In [ ]:
subprocess.run([sys.executable, 'scripts/colab_worker.py', '--worker', WORKER, '--queue', 'colab/team-queue.json',
                '--idle-minutes', '60', '--sync-minutes', '10'], check=True)